#  Dependencies


In [1]:
!apt-get install -y poppler-utils
!pip install -q transformers==4.46.1 peft==0.13.2 qdrant-client colpali_engine torch torchvision Pillow gradio kagglehub accelerate bitsandbytes

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (1,505 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 122412 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 68.5 

In [2]:
import os
import glob
import pandas as pd
import kagglehub
import ast
import torch
import gc
from PIL import Image
from colpali_engine.models import ColPaliProcessor, ColPali
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from google.colab import userdata

#Download Dataset & Preprocess



In [3]:
dataset_path = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")
print(f"Dataset downloaded to: {dataset_path}")

csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
csv_path = csv_files[0]
print(f"Loading metadata from: {csv_path}")
df = pd.read_csv(csv_path)

IMAGE_COLUMN = 'image'
TEXT_COLUMN = 'text'

LIMIT = 200
metadata_list = []
found_count = 0

all_jpg_files = glob.glob(os.path.join(dataset_path, "**", "*.jpg"), recursive=True)

image_path_dict = {os.path.basename(path): path for path in all_jpg_files}
print(f"Found {len(image_path_dict)} total images.")

for index, row in df.iterrows():
    if found_count >= LIMIT:
        break

    raw_img_val = str(row[IMAGE_COLUMN])

    if raw_img_val.startswith('['):
        try:
            img_list = ast.literal_eval(raw_img_val)
            if len(img_list) > 0:
                raw_img_val = img_list[0]
        except:
            pass

    img_identifier = os.path.basename(raw_img_val)

    if not img_identifier.endswith('.jpg'):
        img_identifier += '.jpg'

    if img_identifier in image_path_dict:
        real_image_path = image_path_dict[img_identifier]
        real_report_text = str(row[TEXT_COLUMN])

        if pd.isna(real_report_text) or str(real_report_text).strip() == "":
            continue

        metadata_list.append({
            "file_path": real_image_path,
            "document_name": f"MIMIC_CXR_{img_identifier.split('.')[0]}",
            "report_text": real_report_text
        })
        found_count += 1

print(f"Successfully paired {len(metadata_list)} real X-rays with their actual clinical reports.")

Using Colab cache for faster access to the 'mimic-cxr-dataset' dataset.
Dataset downloaded to: /kaggle/input/mimic-cxr-dataset
Loading metadata from: /kaggle/input/mimic-cxr-dataset/mimic_cxr_aug_validate.csv
Found 261137 total images.
Successfully paired 200 real X-rays with their actual clinical reports.


# Models (ColPali & Gemma)



In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
    print("Hugging Face token found. Authenticating...")
except:
    print("Error: HF_TOKEN not found in Colab Secrets.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware utilized: {device}")

print("Loading ColPali.")
colpali_model_name = "vidore/colpali-v1.3"
processor = ColPaliProcessor.from_pretrained(colpali_model_name, token=hf_token)

retrieval_model = ColPali.from_pretrained(
    colpali_model_name,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    token=hf_token
).eval()

gc.collect()
torch.cuda.empty_cache()

print("Loading MedGemma.")
medgemma_model_id = "google/gemma-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

llm_tokenizer = AutoTokenizer.from_pretrained(medgemma_model_id, token=hf_token)

llm_model = AutoModelForCausalLM.from_pretrained(
    medgemma_model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    token=hf_token
)
print("Models loaded successfully!")

Hugging Face token found. Authenticating...
Hardware utilized: cuda
Loading ColPali.


preprocessor_config.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/862M [00:00<?, ?B/s]

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/78.6M [00:00<?, ?B/s]

#Vector Indexing with Qdrant



In [ ]:
import uuid
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")
collection_name = "mimic_cxr_reports"

client.create_collection(
    collection_name,
    vectors_config={
        "colpali": models.VectorParams(
            size=retrieval_model.dim,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM
            )
        )
    }
)

print("Starting vector indexing of X-Rays.")
for meta in metadata_list:
    image = Image.open(meta["file_path"]).convert("RGB")

    batch_images = processor.process_images([image]).to(device)
    with torch.no_grad():
        image_embedding = retrieval_model(**batch_images)[0].to(torch.float32).cpu().numpy()

    client.upsert(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=uuid.uuid4().hex,
                vector={"colpali": image_embedding},
                payload=meta
            )
        ]
    )
print("Indexing complete! Qdrant database is ready.")

#RAG & Generation Functions



In [ ]:
def generate_llm_response(prompt):
    tokenized = llm_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in tokenized.items()}

    outputs = llm_model.generate(**inputs, max_new_tokens=250, temperature=0.3, do_sample=True)

    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True).replace(prompt, "").strip()

def generate_report(uploaded_image):
    try:
        if uploaded_image is None:
            return "Please upload an X-Ray image.", None

        print("\n=== STARTING MODE 1: REPORT GEN ===")
        image = Image.open(uploaded_image).convert("RGB")

        print("1. Processing Image with ColPali.")
        batch_images = processor.process_images([image]).to(device)
        with torch.no_grad():
            query_embedding = retrieval_model(**batch_images)[0].to(torch.float32).cpu().numpy().tolist()

        print("2. Searching Qdrant Database.")
        results = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            using="colpali",
            limit=1,
            with_payload=True
        ).points

        if not results:
            return "No reference cases found.", None

        reference_payload = results[0].payload
        reference_report = reference_payload["report_text"]
        reference_image_path = reference_payload["file_path"]
        print(f"3. Found Match! Reference file: {reference_image_path}")

        prompt = f"""You are an expert radiologist AI. Based on the following retrieved historical case that is visually identical to the patient's X-ray, generate a structured medical report.

Historical Reference Findings: {reference_report}

Generate a report with the following structure:
- FINDINGS:
- IMPRESSION:

Report:"""

        generated_report = generate_llm_response(prompt)
        print("=== MODE 1 COMPLETE ===")
        return generated_report, reference_image_path

    except Exception as e:
        import traceback
        error_msg = f"System Error: {str(e)}\n\n{traceback.format_exc()}"
        print(error_msg)
        raise gr.Error("Check the Colab terminal for the exact error!")


def answer_clinical_question(query):
    try:
        if not query:
            return "Please enter a question.", None

        print(f"\n=== STARTING MODE 2: QA ===")
        print(f"Question asked: {query}")

        print("1. Processing Text Query with ColPali...")
        batch_queries = processor.process_queries([query]).to(device)
        with torch.no_grad():
            query_embedding = retrieval_model(**batch_queries)[0].to(torch.float32).cpu().numpy().tolist()

        print("2. Searching Qdrant Database.")
        results = client.query_points(
            collection_name=collection_name,
            query=query_embedding,
            using="colpali",
            limit=1,
            with_payload=True
        ).points

        if not results:
            return "No relevant context found.", None

        context_payload = results[0].payload
        context_report = context_payload["report_text"]
        reference_image_path = context_payload["file_path"]
        print(f"3. Found Context! Using report from: {reference_image_path}")

        prompt = f"""You are an expert clinical assistant. Answer the medical question accurately based STRICTLY on the retrieved clinical context.

Context: {context_report}
Question: {query}
Answer:"""

        answer = generate_llm_response(prompt)
        print("=== MODE 2 COMPLETE ===")
        return answer, reference_image_path

    except Exception as e:
        import traceback
        error_msg = f"System Error: {str(e)}\n\n{traceback.format_exc()}"
        print(error_msg)
        raise gr.Error("Check the Colab terminal for the exact error!")

#Gradio UI



In [ ]:
import gradio as gr

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🩺 Multi-Modal Chest X-Ray Intelligence System")
    gr.Markdown("**Dual-Mode Architecture:** Image-to-Report Generation & RAG-based Clinical QA using **ColPali** and **Gemma**.")

    with gr.Tabs():
        with gr.TabItem("Mode 1: Report Generation"):
            with gr.Row():
                with gr.Column():
                    input_image = gr.Image(label="Upload Patient Chest X-Ray", type="filepath")
                    generate_btn = gr.Button("Generate Structured Report", variant="primary")
                with gr.Column():
                    output_report = gr.Textbox(label="Generated AI Report (Gemma)", lines=8)
                    reference_img = gr.Image(label="Retrieved Historical Reference Match (ColPali)")

            generate_btn.click(fn=generate_report, inputs=input_image, outputs=[output_report, reference_img])

        with gr.TabItem("Mode 2: Clinical QA (RAG)"):
            with gr.Row():
                with gr.Column():
                    qa_input = gr.Textbox(label="Ask a clinical question about the database")
                    qa_btn = gr.Button("Search & Answer", variant="primary")
                    qa_output = gr.Textbox(label="Generated Answer", lines=5)
                with gr.Column():
                    qa_evidence = gr.Image(label="Retrieved Evidence X-Ray")

            qa_btn.click(fn=answer_clinical_question, inputs=qa_input, outputs=[qa_output, qa_evidence])

demo.launch(debug=True, allowed_paths=[dataset_path, "/kaggle/input"])

# QA generation

In [ ]:
import pandas as pd
from tqdm import tqdm

qa_data = []
num_reports = min(50, len(metadata_list))

for i in tqdm(range(num_reports), desc="Generating QA Pairs"):
    report = metadata_list[i]["report_text"]
    doc_name = metadata_list[i]["document_name"]

    prompt = f"""You are a strict data extractor building a ground-truth benchmark dataset.
Read the following chest X-ray report and generate exactly ONE Question and Answer pair.

STRICT RULES:
1. The Answer MUST be explicitly stated word-for-word in the text.
2. DO NOT guess, diagnose, or infer causes. No external medical knowledge is allowed.
3. If the text says "The lungs are clear", your question should be "What is the condition of the lungs?" and the answer "The lungs are clear."

Format your response EXACTLY like this:
Question: [Your factual question]
Answer: [The exact factual answer from the report]

Report: {report}
"""

    try:
        response = generate_llm_response(prompt)

        if "Question:" in response and "Answer:" in response:
            parts = response.split("Answer:")
            q_part = parts[0].replace("Question:", "").strip()
            a_part = parts[1].strip()

            qa_data.append({
                "Document": doc_name,
                "Original_Report": report,
                "Generated_Question": q_part,
                "Generated_Answer": a_part
            })
    except Exception as e:
        print(f"Skipping {doc_name} due to formatting error.")

df_qa = pd.DataFrame(qa_data)
df_qa.to_csv("my_reliable_vqa_dataset.csv", index=False)
print(f"\nSuccessfully generated {len(df_qa)}  Question Answer pairs!")
print("File saved as 'my_reliable_vqa_dataset.csv'.")

# CLIP comparison

In [ ]:
import torch
import numpy as np
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel

print("Loading PubMedCLIP...")
clip_processor = CLIPProcessor.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32")
clip_model = CLIPModel.from_pretrained("flaviagiammarino/pubmed-clip-vit-base-patch32").eval().to(device)

print("Extracting CLIP embeddings for all images.")
clip_image_vectors = []
for meta in metadata_list:
    img = Image.open(meta["file_path"]).convert("RGB")
    inputs = clip_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        emb = clip_model.get_image_features(**inputs)
        emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
        clip_image_vectors.append(emb.cpu().numpy())
clip_image_matrix = np.vstack(clip_image_vectors)

print("Running Recall@5 Benchmark")
colpali_successes = 0
clip_successes = 0

for i, meta in enumerate(tqdm(metadata_list)):
    correct_image_path = meta["file_path"]
    query_text = meta["report_text"][:200]

    colpali_q = processor.process_queries([query_text]).to(device)
    with torch.no_grad():
        colpali_q_emb = retrieval_model(**colpali_q)[0].to(torch.float32).cpu().numpy().tolist()

    colpali_results = client.query_points(
        collection_name=collection_name, query=colpali_q_emb, using="colpali", limit=5, with_payload=True
    ).points

    if any(res.payload["file_path"] == correct_image_path for res in colpali_results):
        colpali_successes += 1

    clip_q = clip_processor(text=[query_text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        clip_q_emb = clip_model.get_text_features(**clip_q)
        clip_q_emb = (clip_q_emb / clip_q_emb.norm(p=2, dim=-1, keepdim=True)).cpu().numpy()

    similarities = (clip_image_matrix @ clip_q_emb.T).flatten()
    top_5_guesses = np.argsort(similarities)[-5:]

    if i in top_5_guesses:
        clip_successes += 1

total = len(metadata_list)
print("\n" + "="*40)
print("COMPARISON (Recall@5)")
print("="*40)
print(f"ColPali Accuracy:    {(colpali_successes / total) * 100:.2f}%")
print(f"PubMedCLIP Accuracy: {(clip_successes / total) * 100:.2f}%")
print("="*40)